# Task 2: Clustering

Which genetic perturbations show similar effects, if any? Which clustering method(s) best
capture(s) the underlying biology?

Apply different clustering methods to the data, visualize and compare the results. Students
working alone only need to use data from the co-culture condition. Interpret the results
biologically. Do the clusters correspond to what you would expect based on the findings
described in the paper, e.g., the pathways that are discussed?

In [ ]:
# core preprocessing/scanpy + plotting/enrichment-analysis stack
from scmlcourse import preprocessing
from pathlib import Path
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
import gseapy
import numpy as np
import pandas as pd
import gc

In [ ]:
# free memory from any previous run before loading a fresh AnnData
gc.collect()

In [ ]:
data_dir = Path("../../../data").resolve()
out_dir = Path("../plots/task_2")
out_dir.mkdir(exist_ok=True)
# restrict this notebook to the co-culture condition (as required for
# students working alone)
condition = "Co-culture"
out_dir = out_dir/condition
out_dir.mkdir(exist_ok=True)


In [ ]:
# gene sets used later for GSEA-based cluster annotation (MSigDB Hallmark
# pathways); build a gene -> pathway lookup for labeling perturbed genes
#gene_sets = pd.read_excel(data_dir/"41588_2021_779_MOESM3_ESM_suppl.xlsx", sheet_name="Supplementary Table 5", skiprows=4, header=0, usecols=[0,1,2,3], index_col=0).iloc[:30]
#gene_sets = gene_sets.loc[condition]
#gene_sets_dict = {row["Program Description"]:row["Key Features"].split(", ") for n,row in gene_sets.iterrows()}
#gene_sets_dict
gene_sets_dict = gseapy.get_library("MSigDB_Hallmark_2020")
gene_to_set = {}
for s,genes in gene_sets_dict.items():
    gene_to_set.update({g:s for g in genes})
gene_to_set

In [ ]:
# load QC'd data (running QC from scratch if a cached copy isn't available),
# then keep only cells from the chosen condition
#if (data_dir/"qced_data.h5ad").exists():
#    adata = sc.read_h5ad(data_dir/"qced_data.h5ad")
#else:
#    adata = sc.read_h5ad(data_dir/"frangieh/rna.h5ad")
#    adata = preprocessing.run_qc(adata=adata, plots_dir=Path("../plots/qc").resolve(), out_path=data_dir/"qced_data.h5ad", min_genes=20)
adata = sc.read_h5ad("/home/ntbiotech/Downloads/qced_data.h5ad")

adata = adata[adata.obs["perturbation_2"]==condition]
adata

In [ ]:
# filter out underrepresented perturbations (control cells are dropped
# here since they are not a target for clustering by perturbation effect)
print(f"Perturbations before filter: {len(np.unique(adata.obs["perturbation"]))}")
adata = adata[adata.obs["perturbation"]!="control"]
counts_per_perturbation = adata.obs["perturbation"].value_counts()
in_p = counts_per_perturbation[counts_per_perturbation>100].index
adata = adata[(adata.obs["perturbation"].to_numpy()[:,None]==in_p.to_numpy()[None,:]).any(axis=1)]
print(f"Perturbations after filter: {len(np.unique(adata.obs["perturbation"]))}")

In [ ]:
# normalize counts and log-transform before PCA/clustering
sc.pp.normalize_total(adata, target_sum=1e6)
sc.pp.log1p(adata)


We can use PCA to lower the dimensionality of the data.

In [ ]:
sc.pp.pca(adata)
sc.pl.pca_variance_ratio(adata, save=out_dir/"variance_explained.pdf")

We can check if the first PCs capture noise rather than perturbation effect by looking for trends in the plots:

In [ ]:
# color the leading PCs by QC metrics to check whether they capture
# technical noise (ribo/mito content) rather than biological signal
sc.pl.pca(
    adata,
    color=["pct_counts_ribo","pct_counts_ribo", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
    save=out_dir/"pca_noise.pdf"
)

Plotting the mean PC loadings grouped by the perturbations:

In [ ]:
# plot immune cluster
def plot_pca(adata, col="perturbation", pc=None, plotting_method=None):
    """Heatmap of mean PCA loadings per group in `adata.obs[col]`.

    Uses the first `pc` PCs (all of them if `pc` is None, or `range(pc)`
    if an int). With the default `plotting_method`, restricts to
    JAK/IFN/STAT/CD58-related groups and saves the heatmap under `out_dir`.
    Returns `(fig, mean_pc_matrix)`.
    """
    pcs = adata.obsm["X_pca"]
    if pc == None:
        pc = np.arange(pcs.shape[1])
    elif isinstance(pc, int):
        pc = np.arange(pc)
    pcs = pcs[:, pc]
    y = adata.obs[col]
    if plotting_method==None:
        def plotting_method(pc_matrix, y):
            # mean PC vector per group, restricted to JAK/IFN/STAT-pathway
            # perturbations plus CD58 (relevant immune-evasion genes)
            data = [pc_matrix[y==l].mean(axis=0) if sum(y==l)>1 else pc_matrix[y==l] for l in np.unique(y) if (l.startswith("JAK") or l.startswith("IFN") or l.startswith("STAT") or l=="CD58")]
            fig, ax = plt.subplots(1,1,figsize=(10,10))
            ax = sns.heatmap(data,yticklabels=[l for l in np.unique(y) if (l.startswith("JAK") or l.startswith("IFN") or l.startswith("STAT") or l=="CD58")], ax=ax)
            fig.savefig(out_dir/"pca_jakstat_per_perturbation.pdf")
            return fig, data

    return plotting_method(pcs, y)
fig, mean_pcs_pert = plot_pca(adata, pc=10)

In [ ]:
# same as above, but over every perturbation (not just JAK/IFN/STAT/CD58)
def plot_pca(adata, col="perturbation", pc=None, plotting_method=None):
    """Heatmap of mean PCA loadings per group in `adata.obs[col]`, for all groups.

    Same as the previous `plot_pca`, but the default `plotting_method`
    plots every group instead of just the JAK/IFN/STAT/CD58 subset.
    """
    pcs = adata.obsm["X_pca"]
    if pc == None:
        pc = np.arange(pcs.shape[1])
    elif isinstance(pc, int):
        pc = np.arange(pc)
    pcs = pcs[:, pc]
    y = adata.obs[col]
    if plotting_method==None:
        def plotting_method(pc_matrix, y):
            data = [pc_matrix[y==l].mean(axis=0) if sum(y==l)>1 else pc_matrix[y==l] for l in np.unique(y)]
            fig, ax = plt.subplots(1,1,figsize=(100,100))
            ax = sns.heatmap(data,yticklabels=np.unique(y), ax=ax)
            fig.savefig(out_dir/"pca_per_perturbation.pdf")
            return fig, data

    return plotting_method(pcs, y)
fig, mean_pcs_pert = plot_pca(adata, pc=10)

In [ ]:
# use the first 30 pcs to compute a neighbour graph
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)

In [ ]:
# Check for trends from noise
sc.pl.umap(
    adata,
    color=["pct_counts_ribo","pct_counts_ribo", "pct_counts_mt", "pct_counts_mt"],
    size=2,
    save=out_dir/"umap_noise.pdf"
)

In [ ]:
# Leiden clustering at multiple resolutions for comparison
for res in [0.2,0.5, 1, 2]:
    sc.tl.leiden(adata, flavor="igraph", resolution=res, key_added=f"leiden_{res}")

In [ ]:
# map each cell's perturbed gene to its Hallmark pathway (for later cluster
# annotation) and report how many perturbations couldn't be mapped
adata.obs["perturbed_set"] = adata.obs["perturbation"].map(gene_to_set)
(adata.obs["perturbed_set"].isna()).sum()

In [ ]:
sc.pl.umap(
    adata,
    color=[c for c in adata.obs.columns if c.startswith("leiden")],
    save=out_dir/"umap_leiden.pdf")

In [ ]:
def plot_cluster(adata, pert_col="perturbation", cluster_col="cluster"):
    """Plot a perturbation-by-cluster count heatmap and run GSEA per cluster.

    Builds a `pert_col` x `cluster_col` contingency heatmap, then for each
    cluster runs a pre-ranked GSEA (`gseapy.prerank`, ranked by cell counts
    per perturbation) against `gene_sets_dict`, saving the enrichment and
    dotplot figures under `out_dir`. Returns `(fig, clusters)` where
    `clusters` is the count table.
    """
    clusters = adata.obs[[cluster_col, pert_col]].value_counts().reset_index().pivot(index=pert_col, columns=cluster_col, values="count")
    # norm by pert and cluster counts
    #clusters = clusters/clusters.sum(axis=1).values[:,None]
    #clusters = clusters/clusters.sum(axis=0).values[None,:]
    fig, ax = plt.subplots(1,1,figsize=(10,100))
    ax = sns.heatmap(clusters, ax=ax)
    #fig.savefig(f"../plots/cluster_{cluster_col}.png")

    # top genes per cluster
    for c in clusters.columns:
        ranked = clusters[c].sort_values(ascending=False)
        print(c, ranked[:5], sep="\n")
        gene_set = "GO_Biological_Process_2026"
        gene_set = gene_sets_dict
        enr = gseapy.prerank(
            ranked,
            gene_sets=gene_set,
            organism="human",
            outdir=None,
            seed=0,
            min_size=1,
            max_size=1000
        )

        if enr.res2d.empty:
            print(f"No enriched terms for cluster {c}, skipping.")
            continue

        terms = enr.res2d.Term
        fig1 = enr.plot(
            terms=terms[:5],
            show_ranking=True,
        )
        fig1.savefig(out_dir/f"gsea_plot_{str(gene_set)[:20]}_{pert_col}_{cluster_col}-cluster_{c}.pdf")
        
        plt.show()
        plt.close(fig1)
        ax = gseapy.dotplot(
            enr.res2d,
            column="FDR q-val",
            cmap=plt.cm.viridis,
            size=3,
            cutoff=1,
            show_ring=False,
            fontsize=()
        )
        ax.figure.savefig(out_dir/f"dotplot_{str(gene_set)[:20]}_{pert_col}_{cluster_col}-cluster_{c}.pdf")
        plt.show()
        plt.close(ax.figure)

        #dplot = gseapy.dotplot(enr.res2d, title=f'cluster {c}', cmap='viridis', size=5, figsize=(3, 5))
        
        #return enr.results[enr.results["Adjusted P-value"]<0.05]
        
    return fig, clusters

# annotate leiden_0.2 clusters via GSEA over the perturbed genes they contain
plot_cluster(adata, "perturbation", cluster_col="leiden_0.2")

In [ ]:
# Alternative plot_cluster (row/col-normalized counts, enrichr instead of
# prerank), kept for reference:
#def plot_cluster(adata, pert_col="perturbation", cluster_col="cluster"):
#    clusters = adata.obs[[cluster_col, pert_col]].value_counts().reset_index().pivot(index=pert_col, columns=cluster_col, values="count")
#    # norm by pert and cluster counts
#    clusters = clusters/clusters.sum(axis=1).values[:,None]
#    clusters = clusters/clusters.sum(axis=0).values[None,:]
#    #fig, ax = plt.subplots(1,1,figsize=(10,100))
#    #ax = sns.heatmap(clusters, ax=ax)
#    #fig.savefig(f"../plots/cluster_{cluster_col}.png")
#
#    # top genes per cluster
#    for c in clusters.columns:
#        print(c, 
#            clusters[c].sort_values(ascending=False)[:10],
#             sep="\n")
#        enr = gseapy.enrichr(
#                clusters[c].sort_values(ascending=False)[:30].index.to_list(),
#                gene_sets=['GO_Biological_Process_2026'],
#                organism="human",
#                outdir=None
#            )
#    
#        dplot = gseapy.dotplot(enr.res2d, title=f'cluster {c}', cmap='viridis', size=5, figsize=(3, 5), cutoff=1)
#        plt.show()
#        plt.close(dplot.figure)
#        
#    return fig, clusters
#
#plot_cluster(adata, "perturbed_gene", cluster_col="leiden_0.2")

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
# KMeans clustering directly on the PCA embedding, for a range of k
for n_clusters in range(2, 15):
    kmeans = KMeans(n_clusters=n_clusters)
    adata.obs[f"kmeans_{n_clusters}"] = kmeans.fit_predict(adata.obsm["X_pca"][:,:]).astype(str)


In [ ]:
# compare a subset of KMeans solutions (k=3..6) against the Leiden
# clusterings on the UMAP embedding
ax = sc.pl.umap(
    adata,
    color=[f"kmeans_{n_clusters}" for n_clusters in range(3, 7)]+[c for c in adata.obs.columns if c.startswith("leiden")],
    size=2,
    show=False
)
plt.savefig(out_dir/"umap_clusters.png")

In [ ]:
# annotate the k=5 KMeans clusters via GSEA, same as for leiden_0.2 above
plot_cluster(adata, cluster_col="kmeans_5")

In [ ]:

adata.write_h5ad(data_dir/f"clustered_data_{condition}.h5ad")